<a href="https://colab.research.google.com/github/M4rck0/Datos_Masivos/blob/main/Tarea_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os, glob

In [2]:
spark = (
    SparkSession.builder
    .appName("SparkEnColab")
    .master("local[*]")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

In [4]:
# Leer zip y descomprimir
zip_name = "reddit_technology.zip"
extract_dir = "reddit_extraido"
!unzip -q -o {zip_name} -d {extract_dir}

In [22]:
# Leer parquet
parquets = glob.glob(f"{extract_dir}/**/*.parquet", recursive=True)
ruta_parquet = os.path.commonpath([os.path.dirname(p) for p in parquets])
df = (spark.read
      .option("recursiveFileLookup", "true")
      .parquet(ruta_parquet)
)

# Limpieza (eliminados y nulos)
df = (df
    .filter(~((F.col("author") == "[deleted]") |
              (F.col("body").isin("[deleted]", "[removed]"))))
    .filter(F.col("score").isNotNull())
    .select("created_utc", "author", "score", "body")
)

In [ ]:
print("Columnas:", df.columns)

In [ ]:
print("Filas leídas:", df.count())

In [ ]:
df.show(5, truncate=80)

In [ ]:
# Manipulación columnas (seleccionar, renombrar y reordenar)
df1 = (df
       .withColumnRenamed("author", "autor")
       .withColumnRenamed("body", "mensaje")
       .select("created_utc", "autor", "score", "mensaje")
      )

df1.show(5, truncate=80)

In [ ]:
# Modificar datos (minúsculas)
df2 = (df1
       .withColumn("mensaje_minusculas", F.lower(F.col("mensaje")))
      )

df2.select("mensaje", "mensaje_minusculas").show(3, truncate=80)

In [ ]:
# Agregar columnas calculadas
df3 = (df2
       # Tiempo
       .withColumn("fecha_hora", F.from_unixtime(F.col("created_utc")).cast("timestamp"))
       .withColumn("fecha", F.to_date(F.col("fecha_hora")))
       .withColumn("hora", F.hour(F.col("fecha_hora")))

       # Texto (longitud y número de palabras)
       .withColumn("longitud_mensaje", F.length(F.col("mensaje")))
       .withColumn("num_palabras", F.size(F.split(F.trim(F.col("mensaje")), r"\s+")))

       # Etiqueta del score
       .withColumn(
           "etiqueta_score",
           F.when(F.col("score") < 0, F.lit("negativo"))
            .when(F.col("score") == 0, F.lit("neutro"))
            .otherwise(F.lit("positivo"))
       )
      )

df3.select("autor", "fecha_hora", "fecha", "hora",
           "score", "etiqueta_score", "longitud_mensaje", "num_palabras",
           "mensaje").show(5, truncate=80)

In [ ]:
# Filtrar filas (comentarios con score alto y texto corto)
df_filtrado = (df3
               .filter((F.col("score") >= 50) & (F.col("longitud_mensaje") < 10))
               .orderBy(F.col("score").desc())
              )

print("Filas filtradas:", df_filtrado.count())

df_filtrado.select(
    "autor", "score", "longitud_mensaje", "fecha_hora", "mensaje"
).show(10, truncate=120)

In [ ]:
# Mejores autores por score (agrupación)
top_autores = (df3
               .groupBy("autor")
               .agg(
                   F.count("*").alias("num_comentarios"),
                   F.sum("score").alias("puntaje_total"),
                   F.avg("score").alias("puntaje_promedio")
               )
               .orderBy(F.col("puntaje_total").desc())
              )

top_autores.show(10, truncate=False)

In [ ]:
# Actividad por día (comentarios y score promedio)
df_diario = (df3
           .groupBy("fecha")
           .agg(
               F.count("*").alias("num_comentarios"),
               F.avg("score").alias("puntaje_promedio"),
               F.expr("percentile_approx(score, 0.5)").alias("mediana_puntaje")
           )
           .orderBy("fecha")
          )

df_diario.show(10, truncate=False)